# Test U2 · Istruzione e formato

Notebook Colab per Qwen3-8B, Mistral-Nemo-Instruct-2407, OLMo-2-1124-7B-Instruct e Llama 3.1 8B Instruct. Protocollo: `Protocollo_testU2_istruzione_v2.md` (9 novembre 2026). Versione del notebook: U2-v1-notebook-2 (appaiamento per pair_key).

Nel Test U, p(UNKNOWN) separa C da B in tutti e 4 i modelli solo in prosa (V2). V2 però cambia due cose insieme: il formato e l'istruzione, che contiene una frase di spiegazione della riga incerta. Questo test le separa con un disegno 2 × 2 sulle **stesse 200 storie V2 del Test U** (stesso generatore, stesso seme 20261120):

| Condizione | Formato | Frase che spiega la riga incerta |
|---|---|---|
| P+S | prosa (= V2 del Test U) | sì |
| P−S | prosa | no |
| S+S | simbolico (come ADD) | sì (quella di ADD) |
| S−S | simbolico | no |

**Ipotesi.** HI1: su P−S, W(p(UNKNOWN), C > B) ≥ 0,60 con limite inferiore > 0,5. HI2: W(P+S) − W(P−S) ≥ 0,05 con limite inferiore > 0. HI3: W(P−S) − W(S−S), stessi criteri. I contrasti usano solo le coppie (stessa `pair_key`) presenti in entrambi i bracci, l'interazione quelle presenti in tutti e quattro; poi media per storia. Bootstrap: 2.000 ricampionamenti delle storie. Lettura per la libreria: almeno 3 conferme sui 4 modelli.

**Allineamento.** P+S deve coincidere con V2 del Test U: stesso hash dei prompt, 24 valori di p(UNKNOWN) per modello entro 1e-3, e (solo nella raccolta completa) lo stesso W entro 0,005. Se un controllo fallisce, il modello si ferma e non entra nei conteggi.

**Come partire.**
1. In Colab seleziona una GPU A100 (Runtime → Cambia tipo di runtime).
2. Per la prova lascia `MODALITA = "prova"` e `MODELLI = ["qwen"]`, metti `AVVIA_GPU = True` ed esegui tutte le celle. Dura pochi minuti.
3. Per la raccolta completa metti `MODALITA = "completo"` e `MODELLI = ["qwen", "mistral", "olmo", "llama"]`. Servono circa 45–60 minuti per modello.
4. Llama usa il Secret `HF_TOKEN` già impostato per il Test U.

I risultati vanno su Drive in `MyDrive/testU2_parziali/<modalità>/`. Lo ZIP finale è `testU2_risultati.zip`. Se la sessione si interrompe, rilanciando si riprende dai blocchi già salvati.

In [ ]:
# 1. Configurazione. Il preflight dei dati si può eseguire anche su CPU.
MODALITA = "prova"  # "prova" oppure "completo"
MODELLI = ["qwen"]  # raccolta completa: ["qwen", "mistral", "olmo", "llama"]
AVVIA_GPU = False
USA_DRIVE = True
BATCH_SIZE = 1        # 1 come nei test 4–6 e nel Test U: nessun padding
# transformers: si usa la versione già installata in Colab, come nel Test U (finisce nel manifest di ogni modello).

In [ ]:
import hashlib
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

LABELS = ("YES", "NO", "UNKNOWN")
CONDITIONS = ("PS", "PN", "SS", "SN")          # P+S, P−S, S+S, S−S
NOMI = {"PS": "P+S", "PN": "P−S", "SS": "S+S", "SN": "S−S"}
N_BOOT = 2000
SEED = 20261108                                # stesso seme del bootstrap del Test U

def encode(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=False, allow_nan=False).encode("utf-8")

def fingerprint(obj):
    return hashlib.sha256(encode(obj)).hexdigest()

def write_json(path, obj):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(encode(obj))
    os.replace(tmp, path)

def parse_label(text):
    match = re.match(r"""^\s*[*#`"']*\s*(YES|NO|UNKNOWN)\b""", text, re.IGNORECASE)
    return match.group(1).upper() if match else "ALTRO"

def bootstrap_per_story(per_story, smoke):
    # per_story: Series indicizzata per storia (già mediata per storia)
    x = per_story.to_numpy(float)
    n = len(x)
    if not n:
        return dict(storie=0, stima=None, ci95=None)
    ci = None
    if not smoke and n > 1:
        rng = np.random.default_rng(SEED)
        b = x[rng.integers(0, n, (N_BOOT, n))].mean(axis=1)
        ci = np.quantile(b, [.025, .975]).tolist()
    return dict(storie=n, stima=float(x.mean()), ci95=ci)

def verdict(stat, null, minimum, smoke=False):
    if smoke:
        return "PROVA TECNICA"
    if stat["ci95"] is None:
        return "NON VALUTABILE"
    lo, hi = stat["ci95"]
    if lo > null and stat["stima"] >= minimum:
        return "CONFERMATA"
    return "NON CONFERMATA" if hi <= null else "NON CONCLUSIVA"

In [ ]:
# Generatore delle storie del Test U, invariato (stesso seme 20261120). L'unica aggiunta salva in SYM la versione simbolica delle storie V2.
import itertools, json, hashlib, collections
import numpy as np
from pathlib import Path
DATA_SEED = 20261120
NOUNS = ['kiln', 'lantern', 'canoe', 'ledger', 'trolley', 'beacon', 'quiver', 'easel', 'turbine', 'hammock', 'compass', 'tripod',
         'satchel', 'kettle', 'anvil', 'barrel', 'sundial', 'telescope', 'harp', 'wagon', 'loom', 'drum', 'saddle', 'chimney',
         'furnace', 'orchard', 'pier', 'vault', 'archive', 'workshop', 'garage', 'stable', 'greenhouse', 'bakery', 'quarry',
         'harbor', 'mill', 'depot', 'pantry', 'cellar', 'attic', 'balcony', 'corridor', 'lobby', 'terrace', 'courtyard', 'gazebo',
         'pavilion', 'scanner', 'printer', 'router', 'battery', 'antenna', 'monitor', 'generator', 'conveyor', 'crane', 'pump',
         'ferry', 'tractor', 'bicycle', 'scooter', 'trailer', 'caravan', 'glider', 'kayak', 'raft', 'sled', 'carousel', 'fountain',
         'statue', 'mural', 'banner', 'flag', 'tent', 'cabin', 'shed', 'barn', 'silo', 'windmill', 'lighthouse', 'bridge', 'tunnel',
         'dam', 'well', 'cistern', 'boiler', 'radiator', 'heater', 'freezer', 'oven', 'stove', 'grill', 'blender', 'mixer',
         'toaster', 'kettledrum', 'piano', 'violin', 'cello', 'organ', 'trumpet', 'flute', 'clock', 'mirror', 'cabinet', 'dresser',
         'wardrobe', 'bench', 'stool', 'ladder', 'scaffold', 'elevator', 'escalator', 'turnstile', 'gate', 'fence', 'hedge']
PARTS = ['polished', 'inspected', 'unlocked', 'labeled', 'refueled', 'calibrated', 'swept', 'repainted', 'insured', 'registered',
         'sanitized', 'upgraded', 'serviced', 'cleared', 'certified', 'measured', 'tagged', 'photographed', 'reserved', 'approved']
INSTR = ('Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen '
         'independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.')
INSTR_PROSA = ('Read the short policy and the facts below. When a fact says that something may be true or false, either case is '
               'possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.')
POLS = {'granted': ('Is approval granted?', False), 'P1': ('Is approval refused?', True), 'P2': ('Is approval not granted?', True),
        'P3': ('Is the request rejected?', True)}           # (domanda, la risposta corretta si inverte?)
FAMS = {'ADD': ['AND', 'OR', 'IMPLIES'], 'V1': ['NAND', 'NOR', 'CONV'], 'V2': ['AND', 'OR', 'IMPLIES'], 'POL': ['AND', 'OR', 'IMPLIES'],
        'SCR': ['AND', 'OR', 'IMPLIES']}
N_ST = {'ADD': 120, 'V1': 200, 'V2': 200, 'V3': 60, 'POL': 100, 'SCR': 60}

def rule_fn(fam, neg):
    nb = (lambda b: not b) if neg else (lambda b: b)
    return {'AND': lambda a, b: a and nb(b), 'OR': lambda a, b: a or nb(b), 'IMPLIES': lambda a, b: (not a) or nb(b),
            'NAND': lambda a, b: not (a and nb(b)), 'NOR': lambda a, b: not (a or nb(b)), 'CONV': lambda a, b: (not nb(b)) or a}[fam]
def rule_sym(fam, neg, a, b):
    bb = f'NOT ({b})' if neg else b
    return {'AND': f'({a} AND {bb})', 'OR': f'({a} OR {bb})', 'IMPLIES': f'({a} IMPLIES {bb})',
            'NAND': f'NOT ({a} AND {bb})', 'NOR': f'NOT ({a} OR {bb})', 'CONV': f'({bb} IMPLIES {a})'}[fam]
def classify(f, a_vals, b_vals):
    outs = {bool(f(a, b)) for a in a_vals for b in b_vals}
    return 'YES' if outs == {True} else 'NO' if outs == {False} else 'UNKNOWN'
def flip(ans, do): return {'YES': 'NO', 'NO': 'YES', 'UNKNOWN': 'UNKNOWN'}[ans] if do else ans
fmt = lambda vals: ' or '.join(str(v).lower() for v in vals)
def phrase(name, val):                  # prosa: name = (sostantivo, participio). B e C differiscono solo per "true" / "false":
    n, p = name[0].replace('_', ' '), name[1]   # nessun "not" in più nei fatti (stesso numero di token, controllato in cella 4)
    if len(val) == 2: return f'It may be true or false that the {n} is {p}.'
    return f'It is {"true" if val[0] else "false"} that the {n} is {p}.'
def rule_prose(fam, neg, A, B):
    na, pa, nb_, pb = A[0], A[1], B[0], B[1]
    lit = lambda pos: f'the {nb_} is {pb}' if pos else f'the {nb_} is not {pb}'
    if fam == 'AND': return f'Approval is given if and only if both the {na} is {pa} and {lit(not neg)}.'
    if fam == 'OR': return f'Approval is given if and only if at least one of these holds: the {na} is {pa}; {lit(not neg)}.'
    if fam == 'IMPLIES': return f'Approval is given in every case except one: when the {na} is {pa} and {lit(neg)}.'
    raise ValueError(fam)

SYM = {}   # Test U2: (storia, variante, tipo, posizione) -> (regola simbolica, righe dei fatti)

def build_all():
    SYM.clear()
    rng = np.random.default_rng(DATA_SEED)
    names = [(n, p) for n in NOUNS for p in PARTS]; rng.shuffle(names); taken = set()
    def three():
        out = []
        for nm in names:
            if nm in taken or any(nm[0] == o[0] for o in out): continue
            out.append(nm)
            if len(out) == 3: break
        assert len(out) == 3, 'nomi finiti'
        taken.update(out); return out
    rows = []
    def add(**r): rows.append(r)
    for S in ('ADD', 'V1', 'V2', 'POL', 'SCR'):
        fams = FAMS[S]; n = N_ST[S]
        for i in range(n):
            fam = fams[i % len(fams)]; sid = f'{S}{i+1:03d}'
            A, B, Dn = three(); a, b, d = ('_'.join(A), '_'.join(B), '_'.join(Dn))
            d_val, a_single = bool(rng.integers(2)), bool(rng.integers(2))
            for neg in (False, True):
                f = rule_fn(fam, neg); variant = 'NOT' if neg else 'POS'
                bC = [bv for bv in (False, True) if classify(f, [False, True], [bv]) == 'UNKNOWN']
                bB = [bv for bv in (False, True) if classify(f, [False, True], [bv]) != 'UNKNOWN']
                assert len(bC) == len(bB) == 1; bC, bB = bC[0], bB[0]
                kinds = {'A': {a: [a_single], b: [bB], d: [d_val]}, 'B': {a: [False, True], b: [bB], d: [d_val]},
                         'C': {a: [False, True], b: [bC], d: [d_val]}}
                if S == 'SCR': kinds.pop('C')
                positions = [('ultima', [b, d, a])] if S in ('POL', 'SCR') else [('prima', [a, b, d]), ('ultima', [b, d, a])]
                pols = {'ADD': ['granted'], 'V1': ['granted'], 'V2': ['granted'], 'POL': ['granted', 'P1', 'P2', 'P3'],
                        'SCR': ['P1', 'P2', 'P3']}[S]
                for kind, al in kinds.items():
                    gold = classify(f, al[a], al[b]); assert (gold == 'UNKNOWN') == (kind == 'C')
                    for position, order in positions:
                        for pol in pols:
                            q, inv = POLS[pol]
                            if S == 'V2':
                                nm = {a: A, b: B, d: Dn}
                                lines = [phrase(nm[k], al[k]) for k in order]
                                sp = lambda T: (T[0].replace('_', ' '), T[1])
                                prompt = INSTR_PROSA + '\n\nPolicy: ' + rule_prose(fam, neg, sp(A), sp(B)) + ' No other rule applies.' \
                                         + '\nFacts:\n' + '\n'.join(lines) + '\n\nQuestion: Is approval given?'
                                # Test U2: la stessa storia in notazione simbolica (nessuna chiamata in più al generatore casuale)
                                SYM[(sid, variant, kind, position)] = (rule_sym(fam, neg, a, b), [f'{k} = {fmt(al[k])}.' for k in order])
                            else:
                                lines = [f'{k} = {fmt(al[k])}.' for k in order]
                                expr = rule_sym(fam, neg, a, b)
                                if pol == 'P3':
                                    rule = f'In this fictional policy, the request is rejected if and only if NOT ({expr}). No other rule applies.'
                                else:
                                    rule = f'In this fictional policy, approval is granted if and only if {expr}. No other approval rule applies.'
                                prompt = INSTR + '\n\n[R1] ' + rule + '\n[R2] ' + '\n'.join(lines) + '\n\nQuestion: ' + q
                            add(set=S, story_id=sid, family=fam, variant=variant, kind=kind, position=position, polarity=pol,
                                condition=flip(gold, inv), b_value=str(al[b][0]).lower(), fact_lines=lines, prompt=prompt,
                                pair_key=f'{sid}|{variant}|{position}|{pol}')
    for i in range(N_ST['V3']):                      # numerici: una sola variabile x (riga incerta) + distrattore
        sid = f'V3{i+1:03d}'; X, Dn = three()[:2]; x, d = '_'.join(X), '_'.join(Dn); t = 300 + 7 * i; d_val = bool(rng.integers(2))
        for variant, op, f in (('GE', 'at least', lambda v: v >= t), ('LE', 'at most', lambda v: v <= t)):
            vals = {'A': [t + 2] if variant == 'GE' else [t - 2], 'B': [t + 1, t + 2] if variant == 'GE' else [t - 2, t - 1],
                    'C': [t - 1, t] if variant == 'GE' else [t, t + 1]}
            for kind, xv in vals.items():
                outs = {f(v) for v in xv}; gold = 'YES' if outs == {True} else 'NO' if outs == {False} else 'UNKNOWN'
                assert (gold == 'UNKNOWN') == (kind == 'C')
                for position, order in (('prima', [x, d]), ('ultima', [d, x])):
                    al = {x: xv, d: [d_val]}; lines = [f'{k} = {fmt(al[k])}.' for k in order]
                    rule = f'In this fictional policy, approval is granted if and only if {x} is {op} {t}. No other approval rule applies.'
                    add(set='V3', story_id=sid, family='NUM', variant=variant, kind=kind, position=position, polarity='granted',
                        condition=gold, b_value='', fact_lines=lines,
                        prompt=INSTR + '\n\n[R1] ' + rule + '\n[R2] ' + '\n'.join(lines) + '\n\nQuestion: Is approval granted?',
                        pair_key=f'{sid}|{variant}|{position}|granted')
    return rows

In [ ]:
# Condizioni del Test U2 e funzioni di analisi.
SPIEGAZIONE_P = "When a fact says that something may be true or false, either case is possible and you are not told which. "
SPIEGAZIONE_S = "Each listed value is allowed; one actual value per field is chosen independently from its allowed values. "
INSTR_PROSA_NS = INSTR_PROSA.replace(SPIEGAZIONE_P, "")
INSTR_NS = INSTR.replace(SPIEGAZIONE_S, "")
assert INSTR_PROSA_NS != INSTR_PROSA and INSTR_NS != INSTR
assert INSTR_PROSA_NS == "Read the short policy and the facts below. Answer with a single word on the first line: YES, NO or UNKNOWN."
assert INSTR_NS == ("Use only the supplied fictional policy and facts. "
                    "Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.")

def symbolic_prompt(instr, expr, lines):
    rule = f"In this fictional policy, approval is granted if and only if {expr}. No other approval rule applies."
    return instr + "\n\n[R1] " + rule + "\n[R2] " + "\n".join(lines) + "\n\nQuestion: Is approval granted?"

def make_conditions(dataset, smoke):
    v2 = [r for r in dataset if r["set"] == "V2"]
    if smoke:                                   # prova: 3 storie per famiglia
        keep = {}
        for r in v2:
            keep.setdefault(r["family"], [])
            if r["story_id"] not in keep[r["family"]] and len(keep[r["family"]]) < 3:
                keep[r["family"]].append(r["story_id"])
        ids = {s for v in keep.values() for s in v}
        v2 = [r for r in v2 if r["story_id"] in ids]
    rows = []
    for r in v2:
        assert r["prompt"].startswith(INSTR_PROSA)
        expr, lines = SYM[(r["story_id"], r["variant"], r["kind"], r["position"])]
        prompts = {"PS": r["prompt"],
                   "PN": INSTR_PROSA_NS + r["prompt"][len(INSTR_PROSA):],
                   "SS": symbolic_prompt(INSTR, expr, lines),
                   "SN": symbolic_prompt(INSTR_NS, expr, lines)}
        for cond, prompt in prompts.items():
            base = {k: v for k, v in r.items() if k not in ("prompt", "set", "fact_lines")}
            rows.append(dict(base, set=cond, prompt=prompt, row_id=f"{cond}|{r['pair_key']}|{r['kind']}"))
    assert len({x["row_id"] for x in rows}) == len(rows) == 4 * len(v2)
    assert len({x["prompt"] for x in rows}) == len(rows)
    return rows

def alignment_rows(dataset, key):
    # dalle storie V2 complete, così il controllo gira identico anche nella prova
    by_key = {(r["pair_key"], r["kind"]): r for r in dataset if r["set"] == "V2"}
    out = []
    for pair_key, kind, value in REF_V2[key]:
        r = {k: v for k, v in by_key[(pair_key, kind)].items() if k != "fact_lines"}
        out.append(dict(r, set="PS", row_id=f"ALLINEAMENTO|PS|{pair_key}|{kind}", expected_p_UNKNOWN=value))
    return out

def filter_length(rows, lengths):
    t = pd.DataFrame(rows)
    t["prompt_tokens"] = lengths
    dropped = []
    for _, block in t[t.kind.isin(["B", "C"])].groupby(["set", "pair_key"]):
        assert len(block) == 2
        if block.prompt_tokens.nunique() != 1:
            dropped.extend(block.row_id.tolist())
    removed = set(dropped)
    return [r for r in rows if r["row_id"] not in removed], t[t.row_id.isin(removed)].to_dict("records")

def with_scores(rows):
    t = pd.DataFrame(rows)
    probs = t[[f"p_{k}" for k in LABELS]].to_numpy(float)
    normalized = probs / np.maximum(probs.sum(axis=1, keepdims=True), 1e-300)
    t["meno_p_massima"] = -normalized.max(axis=1)
    t["entropia"] = -(normalized * np.log(np.maximum(normalized, 1e-300))).sum(axis=1)
    t["is_unknown"] = t.parsed.eq("UNKNOWN")
    t["ok"] = t.parsed.eq(t.condition)
    return t

def wide_pairs(t, score):
    b_c = t[t.kind.isin(["B", "C"])]
    w = b_c.pivot(index=["story_id", "pair_key"], columns="kind", values=[score, "is_unknown"]).dropna()
    return w

def pair_wins(w, score, mask=None):
    # vittoria di C su B per coppia, indicizzata per (story_id, pair_key)
    s = w if mask is None else w[mask]
    return (s[score]["C"] > s[score]["B"]).astype(float) + .5 * (s[score]["C"] == s[score]["B"])

def story_wins(w, score, mask=None):
    wins = pair_wins(w, score, mask)
    return wins.groupby(level="story_id").mean(), len(wins)

def condition_metrics(t, smoke):
    w = wide_pairs(t, "p_UNKNOWN")
    wins = pair_wins(w, "p_UNKNOWN")
    per_story, n_pairs = story_wins(w, "p_UNKNOWN")
    W = dict(bootstrap_per_story(per_story, smoke), coppie=n_pairs)
    b_c = t[t.kind.isin(["B", "C"])]
    auroc = float(roc_auc_score(b_c.kind.eq("C"), b_c.p_UNKNOWN)) if b_c.kind.nunique() == 2 else None
    unk = w["is_unknown"].astype(bool)
    equal = {}
    for name, mask in (("entrambi_UNKNOWN", unk.B & unk.C), ("nessuno_UNKNOWN", ~unk.B & ~unk.C),
                       ("senza_UNKNOWN_su_C", ~unk.C)):
        s, n = story_wins(w, "p_UNKNOWN", mask)
        equal[name] = dict(bootstrap_per_story(s, smoke), coppie=n)
    alternative = {}
    for measure in ("meno_p_massima", "entropia"):
        s, n = story_wins(wide_pairs(t, measure), measure)
        alternative[measure] = dict(bootstrap_per_story(s, smoke), coppie=n)
    behavior = [dict(kind=k, n=len(g), quota_unknown=float(g.is_unknown.mean()), acc=float(g.ok.mean()),
                     quota_altro=float(g.parsed.eq("ALTRO").mean()), p_UNKNOWN_mediana=float(g.p_UNKNOWN.median()))
                for k, g in t.groupby("kind")]
    by_position = [dict(kind=k, position=p, n=len(g), quota_unknown=float(g.is_unknown.mean()))
                   for (k, p), g in t.groupby(["kind", "position"])]
    return wins, dict(W=W, AUROC=auroc, W_a_parita_di_risposta=equal, alternative=alternative,
                           comportamento=behavior, comportamento_per_posizione=by_position)

def contrast(wins, signs, smoke):
    # Confronto appaiato per coppia: solo le pair_key (stessa storia, variante e posizione) presenti in tutti i bracci,
    # poi media per storia e bootstrap sulle storie. Il W di ogni condizione (e l'allineamento di P+S) resta sulle sue coppie.
    common = None
    for c in signs:
        common = wins[c].index if common is None else common.intersection(wins[c].index)
    diff = sum(s * wins[c].loc[common] for c, s in signs.items())
    out = bootstrap_per_story(diff.groupby(level="story_id").mean(), smoke)
    out["coppie"] = len(common)
    return out

def analyse(rows, smoke):
    t = with_scores(rows)
    wins, stats = {}, {}
    for cond in CONDITIONS:
        wins[cond], stats[cond] = condition_metrics(t[t["set"] == cond], smoke)
    d = lambda a, b: contrast(wins, {a: 1, b: -1}, smoke)
    hi1 = dict(stats["PN"]["W"], esito=verdict(stats["PN"]["W"], .5, .60, smoke))
    hi2 = dict(d("PS", "PN"), esito=None)
    hi2["esito"] = verdict(hi2, 0, .05, smoke)
    hi3 = dict(d("PN", "SN"), esito=None)
    hi3["esito"] = verdict(hi3, 0, .05, smoke)
    inter = contrast(wins, {"PS": 1, "PN": -1, "SS": -1, "SN": 1}, smoke)
    descr = dict(interazione=inter, PS_meno_SS=d("PS", "SS"), SS_meno_SN=d("SS", "SN"))
    return dict(condizioni=stats, HI1=hi1, HI2=hi2, HI3=hi3, descrittivi=descr)

In [ ]:
# Riferimenti del Test U (revisioni, token delle etichette, prompt e valori di V2). Scritti qui: non serve caricare file.
MODEL_CONFIG = {"qwen": {"id": "Qwen/Qwen3-8B", "sha": "b968826d9c46dd6066d109eabc6255188de91218", "label_ids": {"YES": [7414, 9454, 9693, 9834, 14004, 14080], "NO": [902, 2152, 2308, 2753, 5664, 8996], "UNKNOWN": [9788, 13790, 16088, 21693, 51378, 61395]}}, "mistral": {"id": "mistralai/Mistral-Nemo-Instruct-2407", "sha": "04d8a90549d23fc6bd7f642064003592df51e9b3", "label_ids": {"YES": [13059, 13830, 14842, 16860, 51935, 57445], "NO": [1836, 2649, 3501, 4753, 14089, 16071], "UNKNOWN": [8929, 14183, 35988, 43048, 51641, 117195]}}, "olmo": {"id": "allenai/OLMo-2-1124-7B-Instruct", "sha": "470b1fba1ae01581f270116362ee4aa1b97f4c84", "label_ids": {"YES": [7566, 9642, 9891, 10035, 14331, 14410], "NO": [912, 2201, 2360, 2822, 5782, 9173], "UNKNOWN": [9987, 14109, 16476, 22435, 52478, 62495]}}, "llama": {"id": "meta-llama/Llama-3.1-8B-Instruct", "sha": "0e9e39f249a16976918f6564b8830bc894c89659", "label_ids": {"YES": [7566, 9642, 9891, 10035, 14331, 14410], "NO": [912, 2201, 2360, 2822, 5782, 9173], "UNKNOWN": [9987, 14109, 16476, 22435, 52478, 62495]}}}
HASH_V2_TEST_U = '4a42072411215d30860cd0e159774a50af930e0a9d39866fbd9e261071df5cdd'
W_V2_TEST_U = {"qwen": 0.91375, "mistral": 0.83375, "olmo": 0.73625, "llama": 0.83875}
# 24 valori di p(UNKNOWN) di V2 per modello, dal Test U: (pair_key, tipo, valore)
REF_V2 = {
    'qwen': [["V2001|NOT|prima|granted", "A", 1.9540310702836905e-08], ["V2001|NOT|ultima|granted", "A", 5.3162732882583454e-08], ["V2001|NOT|prima|granted", "B", 0.6212568879127502], ["V2001|NOT|ultima|granted", "B", 0.9998620748519896], ["V2001|NOT|prima|granted", "C", 0.1192108243703842], ["V2001|NOT|ultima|granted", "C", 0.9990869760513306], ["V2003|NOT|prima|granted", "A", 0.0001584525307407], ["V2003|NOT|ultima|granted", "A", 0.000430612679338], ["V2003|NOT|prima|granted", "B", 0.8807953596115112], ["V2003|NOT|ultima|granted", "B", 0.8808000683784485], ["V2003|NOT|prima|granted", "C", 0.991396963596344], ["V2003|NOT|ultima|granted", "C", 0.99928617477417], ["V2002|NOT|prima|granted", "A", 0.181685209274292], ["V2002|NOT|ultima|granted", "A", 0.3158325850963592], ["V2002|NOT|prima|granted", "B", 0.0011696014553308], ["V2002|NOT|ultima|granted", "B", 0.0015013295924291], ["V2002|NOT|prima|granted", "C", 0.7769908905029297], ["V2002|NOT|ultima|granted", "C", 0.5587241053581238], ["V2036|POS|prima|granted", "B", 0.0003353923384565], ["V2191|NOT|ultima|granted", "B", 0.029314387589693], ["V2049|NOT|ultima|granted", "B", 0.6792019605636597], ["V2021|POS|prima|granted", "C", 0.7773127555847168], ["V2102|POS|ultima|granted", "C", 0.9947775602340698], ["V2138|NOT|ultima|granted", "C", 0.999662220478058]],
    'mistral': [["V2001|NOT|prima|granted", "A", 0.5519922375679016], ["V2001|NOT|ultima|granted", "A", 0.3448371589183807], ["V2001|NOT|prima|granted", "B", 0.1508914679288864], ["V2001|NOT|ultima|granted", "B", 0.8997905850410461], ["V2001|NOT|prima|granted", "C", 0.2448613494634628], ["V2001|NOT|ultima|granted", "C", 0.7139725685119629], ["V2003|NOT|prima|granted", "A", 0.0709141045808792], ["V2003|NOT|ultima|granted", "A", 0.0949864983558654], ["V2003|NOT|prima|granted", "B", 0.1720617562532425], ["V2003|NOT|ultima|granted", "B", 0.586043119430542], ["V2003|NOT|prima|granted", "C", 0.6336223483085632], ["V2003|NOT|ultima|granted", "C", 0.829262912273407], ["V2002|NOT|prima|granted", "A", 0.7942593693733215], ["V2002|NOT|ultima|granted", "A", 0.5260875821113586], ["V2002|NOT|prima|granted", "B", 0.6276575922966003], ["V2002|NOT|ultima|granted", "B", 0.90613454580307], ["V2002|NOT|prima|granted", "C", 0.6297875642776489], ["V2002|NOT|ultima|granted", "C", 0.2969728112220764], ["V2058|NOT|prima|granted", "B", 0.1352556496858596], ["V2059|NOT|prima|granted", "B", 0.4490827918052673], ["V2093|POS|prima|granted", "B", 0.7341074347496033], ["V2057|NOT|ultima|granted", "C", 0.5064252614974976], ["V2026|POS|ultima|granted", "C", 0.8431254029273987], ["V2135|POS|ultima|granted", "C", 0.9419180750846864]],
    'olmo': [["V2001|NOT|prima|granted", "A", 1.1788473784690725e-05], ["V2001|NOT|ultima|granted", "A", 0.0022468783427029], ["V2001|NOT|prima|granted", "B", 0.0001253332738997], ["V2001|NOT|ultima|granted", "B", 0.0013311940710991], ["V2001|NOT|prima|granted", "C", 0.0618057250976562], ["V2001|NOT|ultima|granted", "C", 0.2937089502811432], ["V2003|NOT|prima|granted", "A", 0.0122043769806623], ["V2003|NOT|ultima|granted", "A", 0.0084405792877078], ["V2003|NOT|prima|granted", "B", 0.1007031351327896], ["V2003|NOT|ultima|granted", "B", 0.0342822261154651], ["V2003|NOT|prima|granted", "C", 0.0119123505428433], ["V2003|NOT|ultima|granted", "C", 0.0224840082228183], ["V2002|NOT|prima|granted", "A", 0.1275497078895568], ["V2002|NOT|ultima|granted", "A", 0.0612227097153663], ["V2002|NOT|prima|granted", "B", 0.1272782385349273], ["V2002|NOT|ultima|granted", "B", 0.0507776401937007], ["V2002|NOT|prima|granted", "C", 0.0062527488917112], ["V2002|NOT|ultima|granted", "C", 0.0520722121000289], ["V2010|NOT|ultima|granted", "B", 0.0019414164125919], ["V2062|NOT|ultima|granted", "B", 0.0254482366144657], ["V2179|NOT|ultima|granted", "B", 0.0814197808504104], ["V2082|POS|prima|granted", "C", 0.0313234850764274], ["V2119|POS|prima|granted", "C", 0.0912932008504867], ["V2147|POS|prima|granted", "C", 0.2015331238508224]],
    'llama': [["V2001|NOT|prima|granted", "A", 0.6064791679382324], ["V2001|NOT|ultima|granted", "A", 0.4897743165493011], ["V2001|NOT|prima|granted", "B", 0.7758210301399231], ["V2001|NOT|ultima|granted", "B", 0.8333898186683655], ["V2001|NOT|prima|granted", "C", 0.8971620202064514], ["V2001|NOT|ultima|granted", "C", 0.4293145537376404], ["V2003|NOT|prima|granted", "A", 0.0763838738203048], ["V2003|NOT|ultima|granted", "A", 0.18187016248703], ["V2003|NOT|prima|granted", "B", 0.683983564376831], ["V2003|NOT|ultima|granted", "B", 0.796875536441803], ["V2003|NOT|prima|granted", "C", 0.9320769906044006], ["V2003|NOT|ultima|granted", "C", 0.8969063758850098], ["V2002|NOT|prima|granted", "A", 0.7377907633781433], ["V2002|NOT|ultima|granted", "A", 0.7479211688041687], ["V2002|NOT|prima|granted", "B", 0.8447563648223877], ["V2002|NOT|ultima|granted", "B", 0.434302270412445], ["V2002|NOT|prima|granted", "C", 0.9487035870552064], ["V2002|NOT|ultima|granted", "C", 0.8860159516334534], ["V2132|NOT|prima|granted", "B", 0.548170804977417], ["V2038|POS|prima|granted", "B", 0.6996690630912781], ["V2075|POS|prima|granted", "B", 0.7982884645462036], ["V2069|NOT|prima|granted", "C", 0.7763099074363708], ["V2054|POS|prima|granted", "C", 0.8838264346122742], ["V2017|NOT|prima|granted", "C", 0.9403133988380432]],
}

In [ ]:
# 2. Preflight dei dati, senza GPU.
assert MODALITA in ("prova", "completo"), MODALITA
SMOKE = MODALITA == "prova"
DATASET = build_all()
V2_SHA = hashlib.sha256("\n".join(r["prompt"] for r in DATASET if r["set"] == "V2").encode()).hexdigest()
assert V2_SHA == HASH_V2_TEST_U, "Le storie V2 non coincidono con quelle del Test U"
ROWS = make_conditions(DATASET, SMOKE)
DATA_SHA = hashlib.sha256("\n".join(r["prompt"] for r in ROWS).encode()).hexdigest()
if not SMOKE:
    assert len(ROWS) == 9600
print("Modalità", MODALITA, "| V2 identico al Test U:", V2_SHA[:16], "| prompt", len(ROWS), "| dataset", DATA_SHA[:16])
esempio = {c: next(r for r in ROWS if r["set"] == c and r["kind"] == "C") for c in CONDITIONS}
for c, r in esempio.items():
    print(f"\n--- {NOMI[c]} (C, {r['pair_key']}) ---\n{r['prompt']}")

## Inferenza ed esportazione

Per ogni prompt si salvano la risposta generata (8 token, greedy), la risposta letta e la probabilità sul primo token di YES, NO e UNKNOWN (stesse varianti di scrittura del Test U). Prima del benchmark ogni modello riproduce 24 valori di p(UNKNOWN) di V2 del Test U: se lo scarto supera 1e-3, il modello è NON VALUTABILE e si passa al successivo. Nella raccolta completa il W di P+S deve coincidere con quello del Test U entro 0,005.

In [ ]:
if AVVIA_GPU:
    import gc
    import zipfile

    import torch
    import transformers
    from huggingface_hub import HfApi, get_token
    from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
    from transformers import AutoModelForCausalLM, AutoTokenizer

    def hf_token():
        token = get_token()
        if token is None:
            try:
                from google.colab import userdata
                token = userdata.get("HF_TOKEN")
            except Exception:
                token = None
        return token

    def batch_inputs(tokenizer, model_key, prompts, device):
        options = {"enable_thinking": False} if model_key == "qwen" else {}
        texts = [
            tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                          add_generation_prompt=True, **options)
            for p in prompts
        ]
        return tokenizer(texts, padding=True, return_tensors="pt", add_special_tokens=False).to(device)

    def label_ids(tokenizer):
        audit = {}
        for label in LABELS:
            variants = [prefix + word for word in (label, label.capitalize(), label.lower()) for prefix in ("", " ")]
            audit[label] = {v: tokenizer.encode(v, add_special_tokens=False) for v in variants}
        ids = {label: sorted({sequence[0] for sequence in audit[label].values()}) for label in LABELS}
        if len(set().union(*map(set, ids.values()))) != sum(map(len, ids.values())):
            raise RuntimeError(f"Il tokenizer condivide i primi token fra etichette: {ids}")
        return ids, audit

    def batches(rows, size):
        for start in range(0, len(rows), size):
            yield rows[start:start + size]

    def predict(model, tokenizer, key, rows, ids, size):
        for batch in batches(rows, size):
            enc = batch_inputs(tokenizer, key, [r["prompt"] for r in batch], model.device)
            with torch.inference_mode():
                result = model.generate(**enc, do_sample=False, max_new_tokens=8, use_cache=True,
                                        return_dict_in_generate=True, output_logits=True,
                                        pad_token_id=tokenizer.pad_token_id)
                logits = result.logits[0].float()
                if not bool(torch.isfinite(logits).all()):
                    raise RuntimeError("Logit non finiti al primo passo")
                p = logits.softmax(dim=-1)
                scores = torch.stack([p[:, ids[label]].sum(dim=-1) for label in LABELS], dim=-1).cpu().tolist()
                completions = tokenizer.batch_decode(
                    result.sequences[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            lengths = enc["attention_mask"].sum(dim=-1).cpu().tolist()
            for row, text, probability, n in zip(batch, completions, scores, lengths, strict=True):
                yield dict(row, generated=text, parsed=parse_label(text), prompt_tokens=n,
                           mass_total=sum(probability), **{f"p_{k}": v for k, v in zip(LABELS, probability, strict=True)})
            del result, logits, p

    def prompt_lengths(tokenizer, key, rows):
        out = []
        for chunk in batches(rows, 128):
            enc = batch_inputs(tokenizer, key, [r["prompt"] for r in chunk], "cpu")
            out.extend(enc["attention_mask"].sum(dim=-1).tolist())
        return out

    def checkpoint(folder, stage, rows, compute, size=128):
        sig = fingerprint(rows)[:16]
        folder = folder / "checkpoint" / f"{stage}-{sig}"
        folder.mkdir(parents=True, exist_ok=True)
        out = []
        for i, chunk in enumerate(batches(rows, size)):
            path = folder / f"{i:04d}.json"
            expected = [r["row_id"] for r in chunk]
            if path.exists():
                part = json.loads(path.read_text())
            else:
                part = list(compute(chunk))
                if [r["row_id"] for r in part] != expected:
                    raise RuntimeError(f"Risultati incompleti in {stage}, blocco {i}")
                write_json(path, part)
            if [r["row_id"] for r in part] != expected:
                raise RuntimeError(f"Checkpoint incoerente in {stage}, blocco {i}")
            out.extend(part)
            if i == 0 or (i + 1) % 5 == 0 or len(out) == len(rows):
                print(f"  {stage} {len(out)}/{len(rows)}", flush=True)
        return out

    def export_model(folder, result, tables):
        folder.mkdir(parents=True, exist_ok=True)
        write_json(folder / "esito.json", result)
        for name, rows in tables.items():
            path = folder / f"{name}.csv"
            tmp = path.with_suffix(".csv.tmp")
            pd.json_normalize(rows).to_csv(tmp, index=False)
            os.replace(tmp, path)
        names = ["esito.json", *(f"{name}.csv" for name in tables)]
        archive = folder / f"{folder.name}.zip"
        tmp = archive.with_suffix(".zip.tmp")
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
            for n in names:
                z.write(folder / n, n)
        os.replace(tmp, archive)
        return archive

In [ ]:
VERSIONE = "U2-v1-notebook-2"
IPOTESI = ("HI1", "HI2", "HI3")

def evaluate_model(key, config, root, smoke, token):
    manifest = dict(version=VERSIONE, modello=config["id"], sha=config["sha"], dtype="bfloat16", attention="eager",
                    batch_size=BATCH_SIZE, transformers=transformers.__version__, torch=str(torch.__version__),
                    gpu=torch.cuda.get_device_name(0), modalita=MODALITA, seme=20261120, dataset=DATA_SHA)
    folder = root / f"{key}-{fingerprint(manifest)[:12]}"
    folder.mkdir(parents=True, exist_ok=True)
    if (folder / f"{folder.name}.zip").is_file():
        print(f"{key}: risultati già completati")
        return folder
    tokenizer = AutoTokenizer.from_pretrained(config["id"], revision=config["sha"], token=token, trust_remote_code=False,
                                              **({"fix_mistral_regex": True} if key == "mistral" else {}))
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    ids, audit = label_ids(tokenizer)
    if ids != config["label_ids"]:
        raise RuntimeError(f"{key}: token delle etichette diversi dal Test U")
    model = AutoModelForCausalLM.from_pretrained(config["id"], revision=config["sha"], token=token, dtype=torch.bfloat16,
                                                 device_map={"": 0}, attn_implementation="eager",
                                                 trust_remote_code=False, low_cpu_mem_usage=True).eval()
    try:
        return evaluate_loaded(key, model, tokenizer, folder, smoke, manifest, ids)
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

def evaluate_loaded(key, model, tokenizer, folder, smoke, manifest, ids):
    def predictions(rows):
        return predict(model, tokenizer, key, rows, ids, BATCH_SIZE)

    check = list(predict(model, tokenizer, key, alignment_rows(DATASET, key), ids, 1))
    max_delta = max(abs(r["p_UNKNOWN"] - r["expected_p_UNKNOWN"]) for r in check)
    alignment = dict(valori=dict(n=len(check), errore_assoluto_max=max_delta, superato=max_delta <= .001))
    print(f"  Allineamento con V2 del Test U, p(UNKNOWN): max |delta| = {max_delta:.4g}")
    if max_delta > .001:
        print(f"  ATTENZIONE {key}: allineamento non riuscito. Il modello è NON VALUTABILE e si passa al successivo.")
        result = dict(**manifest, label_ids=ids, allineamento=alignment, valutabile=False)
        export_model(folder, result, dict(allineamento=check))
        return folder

    kept, dropped = filter_length(ROWS, prompt_lengths(tokenizer, key, ROWS))
    raw = checkpoint(folder, "benchmark", kept, predictions)
    analysis = analyse(raw, smoke)
    w_ps = analysis["condizioni"]["PS"]["W"]["stima"]
    w_ok = None if smoke else abs(w_ps - W_V2_TEST_U[key]) <= .005
    alignment["W_P+S"] = dict(stima=w_ps, test_U=W_V2_TEST_U[key], superato=w_ok)
    valutabile = w_ok is not False
    if not valutabile:
        print(f"  ATTENZIONE {key}: W di P+S {w_ps:.4f} diverso dal Test U ({W_V2_TEST_U[key]:.4f}). Esiti NON VALUTABILI.")
        for h in IPOTESI:
            analysis[h]["esito"] = "NON VALUTABILE"
    low_mass = float(np.mean([r["mass_total"] < .90 for r in raw]))
    result = dict(**manifest, label_ids=ids, allineamento=alignment, valutabile=valutabile, **analysis,
                  scarti_prompt=len(dropped), quota_massa_bassa=low_mass, cautela_massa=low_mass > .05)
    archive = export_model(folder, result, dict(punteggi=raw, allineamento=check, scarti=dropped))
    print(f"  Salvato {archive.name} | massa < 0.90: {low_mass:.1%} | coppie scartate: {len(dropped) // 2}")
    return folder

def export_total(root, folders, skipped, smoke):
    summaries = {key: json.loads((p / "esito.json").read_text()) for key, p in folders.items()}
    rows, tables = [], []
    for key, r in summaries.items():
        if not r["valutabile"]:
            rows.append(dict(modello=key, condizione="tutte", nota="NON VALUTABILE: allineamento non riuscito"))
            continue
        for c in CONDITIONS:
            s = r["condizioni"][c]
            kinds = {k["kind"]: k for k in s["comportamento"]}
            eq = s["W_a_parita_di_risposta"]
            rows.append(dict(modello=key, condizione=NOMI[c], W=s["W"]["stima"], IC95=s["W"]["ci95"], AUROC=s["AUROC"],
                             coppie=s["W"]["coppie"], W_entrambi_UNK=eq["entrambi_UNKNOWN"]["stima"],
                             W_nessuno_UNK=eq["nessuno_UNKNOWN"]["stima"],
                             UNK_C=kinds["C"]["quota_unknown"], UNK_B=kinds["B"]["quota_unknown"],
                             acc_A=kinds["A"]["acc"], acc_B=kinds["B"]["acc"],
                             W_entropia=s["alternative"]["entropia"]["stima"]))
        table = pd.read_csv(folders[key] / "punteggi.csv", keep_default_na=False)
        table.insert(0, "modello", key)
        tables.append(table)
    hyp = [dict(modello=k, ipotesi=h, stima=r[h]["stima"], IC95=r[h]["ci95"], esito=r[h]["esito"])
           for k, r in summaries.items() if r["valutabile"] for h in IPOTESI]
    scorecard = {h: [k for k, r in summaries.items() if r["valutabile"] and r[h]["esito"] == "CONFERMATA"] for h in IPOTESI}
    h1, h2, h3 = (len(scorecard[h]) >= 3 for h in IPOTESI)
    if smoke:
        reading = "Modalità prova: nessuna lettura scientifica."
    elif h1 and h2:
        reading = ("HI1 e HI2 in almeno 3 modelli: in prosa p(UNKNOWN) supera il criterio anche senza la frase "
                   "che spiega la riga incerta, e la frase migliora comunque W.")
    elif h1:
        reading = ("HI1 in almeno 3 modelli, HI2 no: in prosa p(UNKNOWN) supera il criterio anche senza la frase "
                   "che spiega la riga incerta; un miglioramento dovuto alla frase non è confermato.")
    elif h2:
        reading = "HI2 in almeno 3 modelli, HI1 no: la frase migliora W, mentre P−S resta sotto il criterio di conferma."
    else:
        reading = "HI1 e HI2 sotto 3 conferme: si riporta modello per modello, senza generalizzare."
    if not smoke and h3:
        reading += (" HI3 in almeno 3: il template in prosa senza spiegazione ottiene un W più alto del template "
                    "simbolico senza spiegazione.")
    report = [
        f"Test U2 | modalità {MODALITA} | modelli completati {len(summaries)}/4 | saltati {skipped}",
        "La modalità prova verifica la pipeline e non produce esiti scientifici." if smoke else
        "Regola per la libreria: almeno 3 conferme fra i 4 modelli.",
        "", pd.DataFrame(rows).round(4).to_string(index=False),
        "", pd.DataFrame(hyp).round(4).to_string(index=False) if hyp else "(nessun esito)",
        "", "Conferme: " + json.dumps(scorecard), "Lettura: " + reading,
    ]
    for key, r in summaries.items():
        report.append(f"\n{key}: allineamento {json.dumps(r['allineamento'])}")
        if r["valutabile"]:
            report.append(f"  scarti {r['scarti_prompt']} | massa bassa {r['quota_massa_bassa']:.2%} | "
                          f"interazione {r['descrittivi']['interazione']}")
    write_json(root / "esiti.json", dict(modelli=summaries, conferme=scorecard, lettura=reading, saltati=skipped,
                                         modalita=MODALITA))
    pd.DataFrame(rows).to_csv(root / "condizioni.csv", index=False)
    if tables:
        pd.concat(tables, ignore_index=True).to_csv(root / "punteggi.csv", index=False)
    (root / "rapporto.txt").write_text("\n".join(report), encoding="utf-8")
    archive = root / "testU2_risultati.zip"
    temp = archive.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temp, "w", zipfile.ZIP_DEFLATED) as z:
        for name in ("esiti.json", "condizioni.csv", "rapporto.txt", "punteggi.csv"):
            if (root / name).is_file() and (name != "punteggi.csv" or tables):
                z.write(root / name, name)
    os.replace(temp, archive)
    print("\n".join(report))
    print("ZIP finale:", archive)
    return archive

def run_test():
    if not MODELLI or len(set(MODELLI)) != len(MODELLI) or not set(MODELLI).issubset(MODEL_CONFIG):
        raise ValueError("MODELLI deve contenere chiavi diverse fra qwen, mistral, olmo, llama")
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("Seleziona una GPU A100 in Colab per il protocollo bf16.")
    if MODALITA == "completo" and "A100" not in torch.cuda.get_device_name(0):
        raise RuntimeError("La raccolta completa richiede una GPU A100, come nel Test U.")
    if USA_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    base = Path("/content/drive/MyDrive/testU2_parziali" if USA_DRIVE else "/content/testU2_parziali")
    root = base / MODALITA
    root.mkdir(parents=True, exist_ok=True)
    token = hf_token()
    folders, skipped = {}, {}
    for key in MODELLI:
        print(f"\n{key} | {MODALITA}", flush=True)
        if key == "llama" and not token:
            skipped[key] = "HF_TOKEN assente"
            print(f"  Saltato: {skipped[key]}")
            continue
        try:
            if key == "llama":
                HfApi().auth_check(repo_id=MODEL_CONFIG[key]["id"], token=token)
            folders[key] = evaluate_model(key, MODEL_CONFIG[key], root, MODALITA == "prova", token)
        except HfHubHTTPError as exc:
            code = exc.response.status_code if exc.response is not None else None
            if key != "llama" or (not isinstance(exc, GatedRepoError) and code not in (401, 403)):
                raise
            skipped[key] = "Accesso al checkpoint Llama non autorizzato"
            print(f"  Saltato: {skipped[key]}")
        export_total(root, folders, skipped, MODALITA == "prova")
    return export_total(root, folders, skipped, MODALITA == "prova")

In [ ]:
# 3. Avvio esplicito della prova o della raccolta.
if AVVIA_GPU:
    ARCHIVIO_TEST_U2 = run_test()
else:
    print("Preflight concluso. Imposta AVVIA_GPU = True per eseguire la modalità selezionata.")